# 玉山證券交易數據管理與分析報表

In [9]:
import os
import json
import keyring
import pandas as pd
from datetime import datetime, timedelta
from configparser import ConfigParser
from esun_trade.sdk import SDK
from esun_marketdata.util import TRADE_SDK_ACCOUNT_KEY, TRADE_SDK_CERT_KEY, setup_keyring

# --- 1. 初始化與登入 ---
def login_sdk():
    psd_path = 'psd.txt'
    password = "psd.txt"
    if os.path.exists(psd_path):
        with open(psd_path, 'r', encoding='utf-8') as f:
            p = f.read().strip()
            if p: password = p

    config = ConfigParser()
    config.read('./config.ini')
    account = config['User']['Account']
    setup_keyring(account)
    keyring.set_password(TRADE_SDK_ACCOUNT_KEY, account, password)
    keyring.set_password(TRADE_SDK_CERT_KEY, account, password)
    sdk = SDK(config)
    sdk.login()
    return sdk

try:
    sdk = login_sdk()
    print("🚀 登入成功")
except Exception as e:
    print(f"❌ 登入失敗: {e}")

🚀 登入成功


In [11]:

# 取得歷史成交 (嘗試抓取最近 30 天)
end_date = datetime.now().strftime("%Y-%m-%d")
start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")

try:
    tx_raw = sdk.get_transactions_by_date(start_date, end_date)
except:
    tx_raw = sdk.get_transactions()


In [12]:
tx_raw

[{'buy_sell': 'B',
  'c_date': '20260413',
  'cost': '0',
  'make': '0',
  'make_per': '0.00',
  'price_avg': '2145.00',
  'price_qty': '10725',
  'qty': '5',
  'recv': '0',
  'stk_na': '聯亞',
  'stk_no': '3081',
  's_type': 'O',
  't_date': '20260409',
  'trade': '0',
  'mat_dats': [{'buy_sell': 'B',
    'c_date': '20260413',
    'db_fee': '0',
    'fee': '15',
    'make': '0',
    'make_per': '0.00',
    'order_no': '69489006975550',
    'pay_n': '-10740',
    'price': '2145.00',
    'price_qty': '10725',
    'qty': '5',
    's_type': 'O',
    'stk_na': '聯亞',
    'stk_no': '3081',
    't_date': '20260409',
    't_time': '',
    'tax': '0',
    'tax_g': '0',
    'trade': '0',
    'user_def': ''}]},
 {'buy_sell': 'S',
  'c_date': '20260414',
  'cost': '-6173',
  'make': '743',
  'make_per': '12.04',
  'price_avg': '2315.00',
  'price_qty': '6945',
  'qty': '3',
  'recv': '6916',
  'stk_na': '聯亞',
  'stk_no': '3081',
  's_type': 'O',
  't_date': '20260410',
  'trade': '0',
  'mat_dats': 

In [7]:
# --- 2. 資料存檔 (JSON) ---
doc_dir = 'doc'
json_path = os.path.join(doc_dir, 'transactions.json')
if not os.path.exists(doc_dir): os.makedirs(doc_dir)

print("正在更新交易紀錄至 JSON...")

try:
    # 取得歷史成交 (嘗試抓取最近 30 天)
    end_date = datetime.now().strftime("%Y-%m-%d")
    start_date = (datetime.now() - timedelta(days=30)).strftime("%Y-%m-%d")
    
    try:
        tx_raw = sdk.get_transactions_by_date(start_date, end_date)
    except:
        tx_raw = sdk.get_transactions()

    # 讀取現有 JSON
    history = {}
    if os.path.exists(json_path):
        with open(json_path, 'r', encoding='utf-8') as f:
            history = json.load(f)

    # 更新資料
    for tx in tx_raw:
        date = tx.get('t_date', '未知日期')
        if date not in history: history[date] = []
        
        tx_entry = {
            "stk_no": tx.get('stk_no'),
            "stk_na": tx.get('stk_na'),
            "buy_price": float(tx.get('buy_price', 0)),
            "buy_qty": float(tx.get('buy_qty', 0)),
            "sell_price": float(tx.get('sell_price', 0)),
            "sell_qty": float(tx.get('sell_qty', 0)),
            "profit": float(tx.get('make_a', 0)),
            "total_amount": (float(tx.get('buy_price', 0)) * float(tx.get('buy_qty', 0))) + (float(tx.get('sell_price', 0)) * float(tx.get('sell_qty', 0)))
        }
        
        if tx_entry not in history[date]:
            history[date].append(tx_entry)

    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(history, f, ensure_ascii=False, indent=2)
    
    print(f"✅ 交易紀錄已儲存至 {json_path}")

except Exception as e:
    print(f"❌ 更新 JSON 失敗: {e}")

正在更新交易紀錄至 JSON...
✅ 交易紀錄已儲存至 doc\transactions.json


In [8]:
# --- 3. 時間段報表分析 ---

# [設定查詢時間段]
query_start = "2026-01-01"
query_end = "2026-12-31"

print(f"📊 分析時間段: {query_start} ~ {query_end}")

try:
    with open(json_path, 'r', encoding='utf-8') as f:
        history = json.load(f)
    
    filtered_tx = []
    for date, tasks in history.items():
        clean_date = date.replace('/', '-').replace('.', '-')
        if query_start <= clean_date <= query_end:
            filtered_tx.extend(tasks)
    
    df_history = pd.DataFrame(filtered_tx)
    inv_raw = sdk.get_inventories()
    inv_map = {item.get('stk_no'): item for item in inv_raw}

    report_list = []
    if not df_history.empty:
        for stk_no, group in df_history.groupby('stk_no'):
            stk_na = group['stk_na'].iloc[0]
            buy_amt = (group['buy_price'] * group['buy_qty']).sum()
            sell_amt = (group['sell_price'] * group['sell_qty']).sum()
            cash_profit = group['profit'].sum()
            
            inv_item = inv_map.get(stk_no, {})
            rem_qty = float(inv_item.get('cost_qty', 0))
            rem_avg = abs(float(inv_item.get('cost_sum', 0))) / rem_qty if rem_qty > 0 else 0
            curr_price = float(inv_item.get('price_mkt', 0))
            unrealized = float(inv_item.get('make_a_sum', 0))
            
            report_list.append({
                "編號": stk_no,
                "公司": stk_na,
                "購買金額": buy_amt,
                "賣出金額": sell_amt,
                "現金盈虧": cash_profit,
                "尚餘股數": rem_qty,
                "尚餘均價": rem_avg,
                "現價": curr_price,
                "總盈虧": cash_profit + unrealized
            })

    df_final = pd.DataFrame(report_list)
    if not df_final.empty:
        display(df_final)
        print(f"\n💰 最終累計總盈虧: {df_final['總盈虧'].sum():,.0f} 元")
    else:
        print("⚠️ 該時間段內無交易紀錄")

except Exception as e:
    print(f"❌ 分析報表時發生錯誤: {e}")

📊 分析時間段: 2026-01-01 ~ 2026-12-31
⚠️ 該時間段內無交易紀錄
